# Notebook 03: Quantum Entanglement

---

**Author:** Milan Amrut Joshi  
**Date:** 2026-03-25  
**Prerequisites:** Notebooks 01-02 (Qubits, Gates, Superposition, Measurement)  
**Framework:** Qiskit 1.x

---

## Table of Contents

1. What is Entanglement?
2. Tensor Products and Composite Systems
3. Separable vs Entangled States
4. The CNOT Gate
5. Bell States: The Maximally Entangled Pair
6. Creating All Four Bell States
7. EPR Paradox
8. Bell Inequality and the CHSH Game
9. Verifying Entanglement Correlations
10. Exercises

## 1. What is Entanglement?

**Quantum entanglement** is a phenomenon where two or more particles become correlated in such a way that the quantum state of each particle cannot be described independently — even when the particles are separated by large distances.

### The Key Idea

If two qubits are entangled, measuring one qubit **instantly determines** the state of the other, regardless of the distance between them.

```
NOT Entangled (Separable):        Entangled:

  Qubit A: |+⟩                     Qubit A & B: |Φ⁺⟩ = (|00⟩+|11⟩)/√2
  Qubit B: |0⟩                     
                                    Measure A → |0⟩: then B MUST be |0⟩
  Measure A: random                 Measure A → |1⟩: then B MUST be |1⟩
  Measure B: always |0⟩            
  (independent!)                    (perfectly correlated!)
```

### Why is Entanglement Special?

1. **Non-classical correlations:** The correlations are stronger than any classical system can produce
2. **No communication:** Entanglement cannot be used to send information faster than light
3. **Resource for computation:** Entanglement is a key resource in quantum algorithms, teleportation, and quantum error correction

### Einstein's Objection

Einstein called entanglement "spooky action at a distance" and argued (in the famous EPR paper, 1935) that quantum mechanics must be incomplete. However, Bell's theorem (1964) and subsequent experiments proved that entanglement is a real physical phenomenon that cannot be explained by hidden variables.

## 2. Tensor Products and Composite Systems

### Combining Quantum Systems

When we have two qubits, the combined state lives in a **tensor product** space:

$$\mathcal{H}_{AB} = \mathcal{H}_A \otimes \mathcal{H}_B$$

If $\mathcal{H}_A$ has dimension $d_A$ and $\mathcal{H}_B$ has dimension $d_B$, then $\mathcal{H}_{AB}$ has dimension $d_A \times d_B$.

For two qubits: $\mathbb{C}^2 \otimes \mathbb{C}^2 = \mathbb{C}^4$.

### Tensor Product of Vectors

$$|\psi\rangle_A \otimes |\phi\rangle_B = |\psi\rangle_A |\phi\rangle_B = |\psi\phi\rangle$$

Explicitly:

$$|0\rangle \otimes |0\rangle = \begin{pmatrix}1\\0\end{pmatrix} \otimes \begin{pmatrix}1\\0\end{pmatrix} = \begin{pmatrix}1\\0\\0\\0\end{pmatrix} = |00\rangle$$

### The Four Computational Basis States for 2 Qubits

$$|00\rangle = \begin{pmatrix}1\\0\\0\\0\end{pmatrix}, \quad |01\rangle = \begin{pmatrix}0\\1\\0\\0\end{pmatrix}, \quad |10\rangle = \begin{pmatrix}0\\0\\1\\0\end{pmatrix}, \quad |11\rangle = \begin{pmatrix}0\\0\\0\\1\end{pmatrix}$$

### General 2-Qubit State

$$|\psi\rangle = \alpha_{00}|00\rangle + \alpha_{01}|01\rangle + \alpha_{10}|10\rangle + \alpha_{11}|11\rangle$$

with $|\alpha_{00}|^2 + |\alpha_{01}|^2 + |\alpha_{10}|^2 + |\alpha_{11}|^2 = 1$.

### Tensor Product of Operators

If $U$ acts on qubit A and $V$ acts on qubit B:

$$(U \otimes V)|\psi\rangle_A|\phi\rangle_B = (U|\psi\rangle_A)(V|\phi\rangle_B)$$

$$U \otimes V = \begin{pmatrix} u_{00}V & u_{01}V \\ u_{10}V & u_{11}V \end{pmatrix}$$

This is a **block matrix** — each element of $U$ is replaced by a scaled copy of $V$.

In [ ]:
# Demonstrate tensor products
import numpy as np

# Basis states
ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

# Tensor products of basis states
ket_00 = np.kron(ket_0, ket_0)
ket_01 = np.kron(ket_0, ket_1)
ket_10 = np.kron(ket_1, ket_0)
ket_11 = np.kron(ket_1, ket_1)

print("=== 2-Qubit Computational Basis States ===")
for name, state in [('|00⟩', ket_00), ('|01⟩', ket_01), ('|10⟩', ket_10), ('|11⟩', ket_11)]:
    print(f"  {name} = {state}")

# Verify orthonormality
print("\n=== Orthonormality Check ===")
basis = [ket_00, ket_01, ket_10, ket_11]
labels = ['00', '01', '10', '11']
for i in range(4):
    for j in range(4):
        ip = np.vdot(basis[i], basis[j])
        if not np.isclose(ip, 0):
            print(f"  ⟨{labels[i]}|{labels[j]}⟩ = {ip.real:.0f}")

# Tensor product of operators: H ⊗ I
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
I = np.eye(2)
HI = np.kron(H, I)  # H on qubit 0, I on qubit 1
IH = np.kron(I, H)  # I on qubit 0, H on qubit 1

print("\n=== Tensor Product of Operators ===")
print(f"H ⊗ I =\n{np.round(HI, 3)}")
print(f"\n(H⊗I)|00⟩ = {np.round(HI @ ket_00, 4)}")
print("= (1/√2)(|00⟩ + |10⟩)")

## 3. Separable vs Entangled States

### Separable (Product) States

A 2-qubit state is **separable** if it can be written as a tensor product:

$$|\psi_{\text{sep}}\rangle = |\psi_A\rangle \otimes |\psi_B\rangle$$

**Example:** $|+0\rangle = |+\rangle \otimes |0\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |10\rangle)$

For separable states, each qubit has an independent description. Measuring qubit A tells you nothing about qubit B.

### Entangled States

A state is **entangled** if it **cannot** be written as a tensor product.

**Example:** $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$

Can we write $\frac{1}{\sqrt{2}}(|00\rangle + |11\rangle) = (\alpha|0\rangle + \beta|1\rangle) \otimes (\gamma|0\rangle + \delta|1\rangle)$?

Expanding: $\alpha\gamma|00\rangle + \alpha\delta|01\rangle + \beta\gamma|10\rangle + \beta\delta|11\rangle$

Matching coefficients:
- $\alpha\gamma = 1/\sqrt{2}$
- $\alpha\delta = 0$ → $\alpha = 0$ or $\delta = 0$
- $\beta\gamma = 0$ → $\beta = 0$ or $\gamma = 0$
- $\beta\delta = 1/\sqrt{2}$

From equations 2 and 4: if $\alpha = 0$, then $\alpha\gamma = 0 \neq 1/\sqrt{2}$. Contradiction!
If $\delta = 0$, then $\beta\delta = 0 \neq 1/\sqrt{2}$. Contradiction!

**No solution exists** → the state is entangled! ✓

### The Schmidt Decomposition Test

Any bipartite state can be written as:

$$|\psi\rangle = \sum_i \lambda_i |u_i\rangle_A |v_i\rangle_B$$

where $\lambda_i \geq 0$ are the **Schmidt coefficients**. The state is:
- **Separable** if only one $\lambda_i \neq 0$ (Schmidt rank = 1)
- **Entangled** if more than one $\lambda_i \neq 0$ (Schmidt rank > 1)

In [ ]:
# Test separability using Schmidt decomposition
import numpy as np

def schmidt_decomposition(state_vector, dim_A=2, dim_B=2):
    """Compute Schmidt decomposition of a bipartite state.
    Returns Schmidt coefficients (singular values)."""
    # Reshape state into a matrix
    matrix = state_vector.reshape(dim_A, dim_B)
    # SVD gives the Schmidt decomposition
    U, S, Vh = np.linalg.svd(matrix)
    return S

def is_entangled(state_vector, dim_A=2, dim_B=2, tol=1e-10):
    """Check if a bipartite state is entangled."""
    coeffs = schmidt_decomposition(state_vector, dim_A, dim_B)
    n_nonzero = np.sum(np.abs(coeffs) > tol)
    return n_nonzero > 1, coeffs

# Test various states
states = {
    '|00⟩': np.array([1, 0, 0, 0], dtype=complex),
    '|+0⟩ = (|00⟩+|10⟩)/√2': np.array([1, 0, 1, 0], dtype=complex) / np.sqrt(2),
    '|Φ⁺⟩ = (|00⟩+|11⟩)/√2': np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2),
    '|Ψ⁺⟩ = (|01⟩+|10⟩)/√2': np.array([0, 1, 1, 0], dtype=complex) / np.sqrt(2),
    '(|00⟩+|01⟩+|10⟩+|11⟩)/2': np.array([1, 1, 1, 1], dtype=complex) / 2,
}

print("=== Entanglement Test via Schmidt Decomposition ===")
print(f"{'State':<35} {'Entangled?':<12} {'Schmidt Coefficients'}")
print("─" * 70)
for name, state in states.items():
    entangled, coeffs = is_entangled(state)
    status = 'ENTANGLED' if entangled else 'Separable'
    print(f"{name:<35} {status:<12} {np.round(coeffs, 4)}")

## 4. The CNOT Gate

The **Controlled-NOT (CNOT or CX)** gate is the most important 2-qubit gate. It is essential for creating entanglement.

### Definition

The CNOT gate flips the **target** qubit if and only if the **control** qubit is $|1\rangle$:

$$\text{CNOT}|a, b\rangle = |a, a \oplus b\rangle$$

where $\oplus$ is addition modulo 2 (XOR).

### Truth Table

```
Input  │ Output  │ Action
───────┼─────────┼────────────────────
 |00⟩  │  |00⟩   │ Control=0: no flip
 |01⟩  │  |01⟩   │ Control=0: no flip
 |10⟩  │  |11⟩   │ Control=1: flip target!
 |11⟩  │  |10⟩   │ Control=1: flip target!
```

### Matrix Representation

$$\text{CNOT} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}$$

### Circuit Symbol

```
Control: ───●───
            │
Target:  ───⊕───
```

### In Terms of Projectors

$$\text{CNOT} = |0\rangle\langle 0| \otimes I + |1\rangle\langle 1| \otimes X$$

This reads: "If control is $|0\rangle$, do nothing to target. If control is $|1\rangle$, apply $X$ to target."

In [ ]:
# CNOT gate demonstration
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# CNOT matrix
CNOT = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
], dtype=complex)

# Verify truth table
basis_states = {
    '|00⟩': np.array([1, 0, 0, 0], dtype=complex),
    '|01⟩': np.array([0, 1, 0, 0], dtype=complex),
    '|10⟩': np.array([0, 0, 1, 0], dtype=complex),
    '|11⟩': np.array([0, 0, 0, 1], dtype=complex),
}

labels = ['|00⟩', '|01⟩', '|10⟩', '|11⟩']

print("=== CNOT Truth Table ===")
for name, state in basis_states.items():
    output = CNOT @ state
    output_idx = np.argmax(np.abs(output))
    print(f"  CNOT{name} = {labels[output_idx]}")

# Qiskit CNOT circuit
print("\n=== CNOT Circuit in Qiskit ===")
qc = QuantumCircuit(2)
qc.cx(0, 1)  # Control=qubit 0, Target=qubit 1
print(qc.draw('text'))

# Apply to |10⟩ (set qubit 0 to |1⟩)
qc2 = QuantumCircuit(2)
qc2.x(0)     # Set qubit 0 to |1⟩
qc2.cx(0, 1) # CNOT
sv = Statevector.from_instruction(qc2)
print(f"\nCNOT|10⟩ = {sv}")
print(f"Probabilities: {sv.probabilities_dict()}")

## 5. Bell States: The Maximally Entangled Pair

The **Bell states** (also called EPR pairs) are the four maximally entangled 2-qubit states. They form an orthonormal basis for the 2-qubit Hilbert space.

### The Four Bell States

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

$$|\Phi^-\rangle = \frac{1}{\sqrt{2}}(|00\rangle - |11\rangle)$$

$$|\Psi^+\rangle = \frac{1}{\sqrt{2}}(|01\rangle + |10\rangle)$$

$$|\Psi^-\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$$

### Derivation: Creating $|\Phi^+\rangle$

Start with $|00\rangle$ and apply H to qubit 0, then CNOT:

$$|00\rangle \xrightarrow{H \otimes I} \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)|0\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |10\rangle)$$

$$\xrightarrow{\text{CNOT}} \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle) = |\Phi^+\rangle$$

The CNOT "copies" the control qubit's value to the target (when the target starts as $|0\rangle$).

### Circuit for All Bell States

```
                      ┌───┐     
 |input₁⟩ ───────────┤ H ├──●──  
                      └───┘  │  
 |input₂⟩ ──────────────────⊕──  

 Input    │ Output Bell State
 ─────────┼──────────────────
 |00⟩     │ |Φ⁺⟩ = (|00⟩+|11⟩)/√2
 |10⟩     │ |Φ⁻⟩ = (|00⟩-|11⟩)/√2
 |01⟩     │ |Ψ⁺⟩ = (|01⟩+|10⟩)/√2
 |11⟩     │ |Ψ⁻⟩ = (|01⟩-|10⟩)/√2
```

### Properties of Bell States

1. **Maximally entangled:** Schmidt coefficients are $(1/\sqrt{2}, 1/\sqrt{2})$
2. **Orthonormal:** $\langle\Phi^+|\Phi^-\rangle = 0$, etc.
3. **Form a basis:** Any 2-qubit state can be written as a linear combination of Bell states
4. **Local indistinguishability:** Measuring just one qubit always gives 50/50 random outcomes

In [ ]:
# Create and verify all four Bell states
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

def create_bell_state(input_state):
    """Create Bell state from input. input_state is '00', '10', '01', or '11'."""
    qc = QuantumCircuit(2)
    # Prepare input
    if input_state[0] == '1':
        qc.x(0)
    if input_state[1] == '1':
        qc.x(1)
    # Bell circuit: H then CNOT
    qc.h(0)
    qc.cx(0, 1)
    return qc

bell_states = {
    '|Φ⁺⟩': '00',
    '|Φ⁻⟩': '10',
    '|Ψ⁺⟩': '01',
    '|Ψ⁻⟩': '11',
}

expected = {
    '|Φ⁺⟩': '(|00⟩ + |11⟩)/√2',
    '|Φ⁻⟩': '(|00⟩ - |11⟩)/√2',
    '|Ψ⁺⟩': '(|01⟩ + |10⟩)/√2',
    '|Ψ⁻⟩': '(|01⟩ - |10⟩)/√2',
}

print("=== All Four Bell States ===")
for name, inp in bell_states.items():
    qc = create_bell_state(inp)
    sv = Statevector.from_instruction(qc)
    print(f"\n{name} = {expected[name]}")
    print(f"  Input: |{inp}⟩")
    print(f"  Statevector: {sv}")
    print(f"  Probabilities: {sv.probabilities_dict()}")
    print(f"  Circuit:")
    print(qc.draw('text'))

In [ ]:
# Verify Bell states are orthonormal
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

def make_bell_sv(input_state):
    qc = QuantumCircuit(2)
    if input_state[0] == '1': qc.x(0)
    if input_state[1] == '1': qc.x(1)
    qc.h(0)
    qc.cx(0, 1)
    return Statevector.from_instruction(qc)

bell_names = ['|Φ⁺⟩', '|Φ⁻⟩', '|Ψ⁺⟩', '|Ψ⁻⟩']
bell_inputs = ['00', '10', '01', '11']
bell_svs = [make_bell_sv(inp) for inp in bell_inputs]

print("=== Bell State Inner Products ===")
print(f"{'':>6}", end='')
for name in bell_names:
    print(f"{name:>8}", end='')
print()

for i, name_i in enumerate(bell_names):
    print(f"{name_i:>6}", end='')
    for j, name_j in enumerate(bell_names):
        ip = np.vdot(bell_svs[i].data, bell_svs[j].data)
        print(f"{ip.real:>8.4f}", end='')
    print()

print("\n→ Diagonal = 1 (normalized), off-diagonal = 0 (orthogonal)")
print("→ Bell states form an orthonormal basis for C⁴!")

## 6. Detailed Derivation of All Four Bell States

### $|\Phi^+\rangle$ from $|00\rangle$

$$|00\rangle \xrightarrow{H_0} \frac{|0\rangle+|1\rangle}{\sqrt{2}} \otimes |0\rangle = \frac{|00\rangle+|10\rangle}{\sqrt{2}} \xrightarrow{\text{CNOT}} \frac{|00\rangle+|11\rangle}{\sqrt{2}} = |\Phi^+\rangle$$

### $|\Phi^-\rangle$ from $|10\rangle$

$$|10\rangle \xrightarrow{H_0} \frac{|0\rangle-|1\rangle}{\sqrt{2}} \otimes |0\rangle = \frac{|00\rangle-|10\rangle}{\sqrt{2}} \xrightarrow{\text{CNOT}} \frac{|00\rangle-|11\rangle}{\sqrt{2}} = |\Phi^-\rangle$$

Note: $H|1\rangle = |-\rangle = (|0\rangle - |1\rangle)/\sqrt{2}$, which introduces the minus sign.

### $|\Psi^+\rangle$ from $|01\rangle$

$$|01\rangle \xrightarrow{H_0} \frac{|0\rangle+|1\rangle}{\sqrt{2}} \otimes |1\rangle = \frac{|01\rangle+|11\rangle}{\sqrt{2}} \xrightarrow{\text{CNOT}} \frac{|01\rangle+|10\rangle}{\sqrt{2}} = |\Psi^+\rangle$$

### $|\Psi^-\rangle$ from $|11\rangle$

$$|11\rangle \xrightarrow{H_0} \frac{|0\rangle-|1\rangle}{\sqrt{2}} \otimes |1\rangle = \frac{|01\rangle-|11\rangle}{\sqrt{2}} \xrightarrow{\text{CNOT}} \frac{|01\rangle-|10\rangle}{\sqrt{2}} = |\Psi^-\rangle$$

### Summary Pattern

The first qubit's initial state controls the **sign** ($\pm$), and the second qubit's initial state controls whether we get **$\Phi$ (same)** or **$\Psi$ (different)**:

| Input | First qubit controls | Second qubit controls | Bell State |
|-------|---------------------|----------------------|------------|
| $|00\rangle$ | + sign | $\Phi$ (00, 11) | $|\Phi^+\rangle$ |
| $|10\rangle$ | - sign | $\Phi$ (00, 11) | $|\Phi^-\rangle$ |
| $|01\rangle$ | + sign | $\Psi$ (01, 10) | $|\Psi^+\rangle$ |
| $|11\rangle$ | - sign | $\Psi$ (01, 10) | $|\Psi^-\rangle$ |

## 7. EPR Paradox

### The Einstein-Podolsky-Rosen Argument (1935)

Einstein, Podolsky, and Rosen (EPR) considered the following thought experiment:

1. Two particles are prepared in the Bell state $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$
2. The particles are sent to two distant observers: Alice and Bob
3. Alice measures her qubit

```
                    Source
                   /      \
                  /  |Φ⁺⟩  \
                 /          \
    ┌─────────┐              ┌─────────┐
    │  Alice  │              │   Bob   │
    │ (Qubit A)│  <───────>  │(Qubit B)│
    │         │  light-years │         │
    └─────────┘              └─────────┘
    
    Alice measures |0⟩  →  Bob's qubit is INSTANTLY |0⟩
    Alice measures |1⟩  →  Bob's qubit is INSTANTLY |1⟩
```

### EPR's Reasoning

1. If Alice measures $|0\rangle$, she **instantly knows** Bob will measure $|0\rangle$
2. This "determination" happens instantly, regardless of distance
3. EPR argued this violates special relativity (no faster-than-light communication)
4. Their conclusion: quantum mechanics is **incomplete** — there must be "hidden variables"

### The Resolution

- Entanglement **does not** transmit information faster than light
- Alice's individual results are random (50/50), and so are Bob's
- The correlations only become apparent when Alice and Bob **compare** their results (which requires classical communication)
- Bell's theorem (1964) proved that no hidden variable theory can reproduce all quantum predictions

In [ ]:
# Simulate EPR experiment: show correlations
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()

# Create Bell state |Φ⁺⟩ and measure both qubits
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

print("EPR Experiment Circuit:")
print(qc.draw('text'))

# Run many times
shots = 10000
result = simulator.run(qc, shots=shots).result()
counts = result.get_counts()

print(f"\n=== EPR Results ({shots} shots) ===")
print(f"Measurement outcomes: {counts}")

# Check correlations
n_same = counts.get('00', 0) + counts.get('11', 0)
n_diff = counts.get('01', 0) + counts.get('10', 0)
print(f"\nAlice and Bob get SAME result: {n_same} ({n_same/shots*100:.1f}%)")
print(f"Alice and Bob get DIFF result: {n_diff} ({n_diff/shots*100:.1f}%)")
print("\n→ 100% correlation! But each individual qubit is random 50/50.")

# Show Alice's marginal statistics
alice_0 = counts.get('00', 0) + counts.get('01', 0)
alice_1 = counts.get('10', 0) + counts.get('11', 0)
print(f"\nAlice's results alone: |0⟩={alice_0} ({alice_0/shots*100:.1f}%), |1⟩={alice_1} ({alice_1/shots*100:.1f}%)")
print("→ Locally random! No information transmitted.")

## 8. Bell Inequality and the CHSH Game

### Bell's Theorem (1964)

John Bell proved that **no local hidden variable theory** can reproduce all the predictions of quantum mechanics. He derived an inequality (now called the **Bell inequality**) that any local realistic theory must satisfy, but quantum mechanics can **violate**.

### The CHSH Inequality

The most experimentally testable form is the **CHSH inequality** (Clauser-Horne-Shimony-Holt, 1969):

$$|S| = |\langle A_0 B_0 \rangle + \langle A_0 B_1 \rangle + \langle A_1 B_0 \rangle - \langle A_1 B_1 \rangle| \leq 2$$

where:
- $A_0, A_1$ are Alice's two measurement choices (different bases)
- $B_0, B_1$ are Bob's two measurement choices
- $\langle A_i B_j \rangle$ is the expectation value of the product of outcomes ($\pm 1$)

### Classical Bound: $|S| \leq 2$

Any local hidden variable theory gives $|S| \leq 2$.

### Quantum Violation: $|S| \leq 2\sqrt{2} \approx 2.828$

Quantum mechanics allows $|S|$ up to $2\sqrt{2}$ (the **Tsirelson bound**).

### Optimal CHSH Strategy

Use the Bell state $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ with measurement angles:

- Alice: $A_0 = Z$ (0°), $A_1 = X$ (90°)
- Bob: $B_0 = \frac{Z+X}{\sqrt{2}}$ (45°), $B_1 = \frac{Z-X}{\sqrt{2}}$ (135°)

```
          Alice's angles        Bob's angles
              
              Z (0°)            (Z+X)/√2 (45°)
              │                      /
              │                    / 
              │                  /
     ─────────┼─── X (90°)    ──/──────── (Z-X)/√2 (-45°)
              │                  \
              │                    \
              │                      \
```

In [ ]:
# Implement CHSH experiment
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np

simulator = AerSimulator()
shots = 50000

def chsh_circuit(alice_angle, bob_angle):
    """Create CHSH experiment circuit.
    Alice and Bob measure in rotated bases specified by angles.
    """
    qc = QuantumCircuit(2, 2)
    # Create Bell state |Φ⁺⟩
    qc.h(0)
    qc.cx(0, 1)
    # Alice rotates her qubit by her angle
    qc.ry(-2 * alice_angle, 0)
    # Bob rotates his qubit by his angle
    qc.ry(-2 * bob_angle, 1)
    # Measure both
    qc.measure([0, 1], [0, 1])
    return qc

def compute_expectation(counts, shots):
    """Compute ⟨AB⟩ from measurement counts.
    Map outcomes: 0 → +1, 1 → -1.
    ⟨AB⟩ = P(same) - P(different)
    """
    same = counts.get('00', 0) + counts.get('11', 0)
    diff = counts.get('01', 0) + counts.get('10', 0)
    return (same - diff) / shots

# CHSH optimal angles
# Alice: 0, π/4
# Bob: π/8, -π/8 (equivalently 3π/8)
alice_angles = [0, np.pi/4]
bob_angles = [np.pi/8, -np.pi/8]

print("=== CHSH Experiment ===")
print(f"Alice's angles: A₀ = 0°, A₁ = 45°")
print(f"Bob's angles:   B₀ = 22.5°, B₁ = -22.5°")
print()

expectations = {}
for i, a in enumerate(alice_angles):
    for j, b in enumerate(bob_angles):
        qc = chsh_circuit(a, b)
        result = simulator.run(qc, shots=shots).result()
        counts = result.get_counts()
        E = compute_expectation(counts, shots)
        expectations[(i, j)] = E
        print(f"  ⟨A{i}B{j}⟩ = {E:.4f} (theory: {np.cos(2*(a-b)):.4f})")

# Compute S = ⟨A₀B₀⟩ + ⟨A₀B₁⟩ + ⟨A₁B₀⟩ - ⟨A₁B₁⟩
S = expectations[(0,0)] + expectations[(0,1)] + expectations[(1,0)] - expectations[(1,1)]
S_theory = 2 * np.sqrt(2)

print(f"\n=== CHSH Value ===")
print(f"  S = {S:.4f}")
print(f"  Theoretical maximum: 2√2 = {S_theory:.4f}")
print(f"  Classical bound: |S| ≤ 2")
print(f"\n  {'✓ BELL INEQUALITY VIOLATED!' if abs(S) > 2 else '✗ No violation'}")
print(f"  S > 2 by {abs(S) - 2:.4f} — this rules out local hidden variables!")

## 9. Verifying Entanglement Correlations

Let's conduct a comprehensive experiment to verify the correlations in each Bell state when measured in different bases.

In [ ]:
# Comprehensive correlation analysis for all Bell states
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np

simulator = AerSimulator()
shots = 10000

def create_bell_circuit(bell_type):
    """Create a Bell state circuit. bell_type in ['Phi+', 'Phi-', 'Psi+', 'Psi-']"""
    qc = QuantumCircuit(2, 2)
    if bell_type in ['Phi-', 'Psi-']:
        qc.x(0)
    if bell_type in ['Psi+', 'Psi-']:
        qc.x(1)
    qc.h(0)
    qc.cx(0, 1)
    return qc

print("=== Correlation Analysis for All Bell States ===")
print("="*60)

for bell_type in ['Phi+', 'Phi-', 'Psi+', 'Psi-']:
    qc = create_bell_circuit(bell_type)
    qc.measure([0, 1], [0, 1])
    
    result = simulator.run(qc, shots=shots).result()
    counts = result.get_counts()
    
    p00 = counts.get('00', 0) / shots
    p01 = counts.get('01', 0) / shots
    p10 = counts.get('10', 0) / shots
    p11 = counts.get('11', 0) / shots
    
    print(f"\n|{bell_type}⟩:")
    print(f"  P(00) = {p00:.4f}, P(01) = {p01:.4f}, P(10) = {p10:.4f}, P(11) = {p11:.4f}")
    
    if bell_type.startswith('Phi'):
        print(f"  → Correlated: P(same) = {p00+p11:.4f}, P(diff) = {p01+p10:.4f}")
    else:
        print(f"  → Anti-correlated: P(same) = {p00+p11:.4f}, P(diff) = {p01+p10:.4f}")

In [ ]:
# Visualize: measure Bell state in different bases
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt
import numpy as np

simulator = AerSimulator()
shots = 10000

def bell_measure_rotated(angle_a, angle_b):
    """Create |Φ⁺⟩ and measure in rotated bases."""
    qc = QuantumCircuit(2, 2)
    # Create |Φ⁺⟩
    qc.h(0)
    qc.cx(0, 1)
    # Rotate measurement bases
    qc.ry(-2 * angle_a, 0)
    qc.ry(-2 * angle_b, 1)
    qc.measure([0, 1], [0, 1])
    return qc

# Vary Bob's angle while Alice measures in Z-basis (angle=0)
bob_angles = np.linspace(0, 2*np.pi, 40)
correlations = []

for angle_b in bob_angles:
    qc = bell_measure_rotated(0, angle_b)
    result = simulator.run(qc, shots=shots).result()
    counts = result.get_counts()
    same = counts.get('00', 0) + counts.get('11', 0)
    diff = counts.get('01', 0) + counts.get('10', 0)
    E = (same - diff) / shots
    correlations.append(E)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(np.degrees(bob_angles), correlations, 'b-', linewidth=2, label='Observed ⟨AB⟩')
ax.plot(np.degrees(bob_angles), np.cos(2*bob_angles), 'r--', linewidth=1.5, label='Theory: cos(2θ_B)')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel("Bob's measurement angle (degrees)", fontsize=13)
ax.set_ylabel('Correlation ⟨AB⟩', fontsize=13)
ax.set_title('Bell State |Φ⁺⟩: Correlation vs Measurement Angle', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 360])
ax.set_ylim([-1.1, 1.1])
plt.tight_layout()
plt.show()

print("When Bob's angle = 0°: perfect correlation (⟨AB⟩ = 1)")
print("When Bob's angle = 45°: no correlation (⟨AB⟩ = 0)")
print("When Bob's angle = 90°: perfect anti-correlation (⟨AB⟩ = -1)")

In [ ]:
# Entanglement entropy: quantifying entanglement
import numpy as np

def von_neumann_entropy(state_vector, dim_A=2, dim_B=2):
    """Compute von Neumann entropy of the reduced density matrix.
    For a pure bipartite state, this quantifies entanglement.
    S = 0 → separable, S = log2(min(dA, dB)) → maximally entangled.
    """
    # Schmidt decomposition
    matrix = state_vector.reshape(dim_A, dim_B)
    _, S, _ = np.linalg.svd(matrix)
    
    # Von Neumann entropy: S = -Σ λᵢ² log₂(λᵢ²)
    probs = S**2
    probs = probs[probs > 1e-12]  # Remove zeros
    entropy = -np.sum(probs * np.log2(probs))
    return entropy

# Test states
states = {
    '|00⟩ (separable)': np.array([1, 0, 0, 0], dtype=complex),
    '|+0⟩ (separable)': np.array([1, 0, 1, 0], dtype=complex) / np.sqrt(2),
    '|Φ⁺⟩ (maximally entangled)': np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2),
    '|Ψ⁻⟩ (maximally entangled)': np.array([0, 1, -1, 0], dtype=complex) / np.sqrt(2),
    'Partially entangled': np.array([np.sqrt(0.8), 0, 0, np.sqrt(0.2)], dtype=complex),
}

print("=== Entanglement Entropy (von Neumann) ===")
print(f"{'State':<40} {'Entropy':>10} {'Entangled?':>12}")
print("─" * 65)
for name, state in states.items():
    S = von_neumann_entropy(state)
    status = 'Maximally' if np.isclose(S, 1.0) else ('Partially' if S > 0.01 else 'No')
    print(f"{name:<40} {S:>10.4f} {status:>12}")

print("\nS = 0: separable (no entanglement)")
print("S = 1: maximally entangled (for 2 qubits)")

## 10. Exercises

### Exercise 1: GHZ State
The **GHZ (Greenberger-Horne-Zeilinger) state** is a 3-qubit entangled state:
$$|\text{GHZ}\rangle = \frac{1}{\sqrt{2}}(|000\rangle + |111\rangle)$$
Create this state using Qiskit and verify the measurement outcomes.

### Exercise 2: W State
The **W state** is another type of 3-qubit entanglement:
$$|W\rangle = \frac{1}{\sqrt{3}}(|001\rangle + |010\rangle + |100\rangle)$$
Research how to create this and implement it.

### Exercise 3: Bell State Discrimination
Given an unknown Bell state, design a circuit that determines which of the four Bell states it is. Hint: Apply CNOT then H (the reverse of Bell state creation).

### Exercise 4: Superdense Coding
Using entanglement, Alice can send 2 classical bits by transmitting only 1 qubit (superdense coding). Implement this protocol.

### Exercise 5: Monogamy of Entanglement
Show that if qubit A is maximally entangled with qubit B, then A cannot be entangled with qubit C at all. Demonstrate this with a 3-qubit circuit.

In [ ]:
# Exercise 1 Solution: GHZ State
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

# Create GHZ state: H on qubit 0, then CNOT cascade
qc = QuantumCircuit(3, 3)
qc.h(0)
qc.cx(0, 1)
qc.cx(0, 2)
print("GHZ State Circuit:")
print(qc.draw('text'))

# Verify statevector
qc_sv = QuantumCircuit(3)
qc_sv.h(0)
qc_sv.cx(0, 1)
qc_sv.cx(0, 2)
sv = Statevector.from_instruction(qc_sv)
print(f"\nStatevector: {sv}")
print(f"Probabilities: {sv.probabilities_dict()}")

# Measure
qc.measure([0, 1, 2], [0, 1, 2])
simulator = AerSimulator()
result = simulator.run(qc, shots=5000).result()
counts = result.get_counts()
print(f"\nMeasurement results: {counts}")
print("→ Only |000⟩ and |111⟩ appear: all qubits are perfectly correlated!")

## Summary

In this notebook, we covered:

1. **Entanglement** — quantum correlations that cannot be explained classically
2. **Tensor products** — mathematical framework for composite quantum systems
3. **Separable vs entangled states** — Schmidt decomposition test
4. **CNOT gate** — the essential 2-qubit gate for creating entanglement
5. **Bell states** — the four maximally entangled 2-qubit states with full derivations
6. **EPR paradox** — Einstein's objection and its resolution
7. **Bell inequality (CHSH)** — quantum mechanics violates classical bounds ($S > 2$)
8. **Correlation verification** — experimental confirmation of entanglement correlations
9. **Entanglement entropy** — von Neumann entropy quantifies entanglement

### Key Takeaway

Entanglement is not just a curiosity — it is a **computational resource** that enables quantum teleportation, superdense coding, quantum error correction, and exponential speedups in algorithms.

### Next Notebook

In **Notebook 04**, we take a deep dive into **quantum circuits** — universal gate sets, controlled operations, circuit compilation, and optimization.